In [1]:
import os

print("===== 현재 Colab 파일 =====")

for file in os.listdir("/content"):
    print(file)

===== 현재 Colab 파일 =====
.config
fraud_full_features.zip
sample_data


In [2]:
import os
import zipfile
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
THRESHOLD = 0.9

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [4]:
import os
import zipfile
import pandas as pd
import numpy as np

zip_path = "/content/fraud_full_features.zip"
extract_path = "/content/fraud_full_features"

# 압축 해제
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("압축 해제 완료")

# CSV 불러오기
df = pd.read_csv(
    "/content/fraud_full_features/fraud_full_features.csv"
)

print("데이터 크기:", df.shape)
print("컬럼 수:", len(df.columns))

압축 해제 완료
데이터 크기: (1296675, 31)
컬럼 수: 31


In [5]:
# ==========================================
# STEP 3. CSV 불러오기
# ==========================================

import pandas as pd

df = pd.read_csv(
    "/content/fraud_full_features/fraud_full_features.csv"
)

print("데이터 로드 완료!")
print("데이터 크기:", df.shape)
print("컬럼 수:", len(df.columns))

데이터 로드 완료!
데이터 크기: (1296675, 31)
컬럼 수: 31


In [6]:
# ==========================================
# STEP 4. 시간 변수 변환 + 시간순 정렬
# ==========================================

df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"]
)

df = (
    df
    .sort_values("trans_date_trans_time")
    .reset_index(drop=True)
)

print("전체 기간:")
print(
    df["trans_date_trans_time"].min(),
    "~",
    df["trans_date_trans_time"].max()
)

print("\n전체 거래 수:", len(df))

전체 기간:
2019-01-01 00:00:18 ~ 2020-06-21 12:13:37

전체 거래 수: 1296675


In [7]:
# ==========================================
# STEP 5. 데이터 기본 확인
# ==========================================

print("데이터 크기:", df.shape)

print("\n상위 5개 행:")
display(df.head())

print("\n타깃 분포:")
print(df["is_fraud"].value_counts())

print(
    "\nFraud Rate (%):",
    df["is_fraud"].mean() * 100
)

데이터 크기: (1296675, 31)

상위 5개 행:


,trans_date_trans_time,cc_num,merchant,category,amt,is_fraud,recent_24h_high_amt_count,category_recent_fraud_rate,category_recent_fraud_rate_missing,count_30min,Repeat3,high_speed,speed_2,customer_mean_amt,customer_std_amt,amt_ratio_to_mean,amt_zscore_card,customer_transaction_count,trans_hour,age,prior_normal_median_amt,amt_to_prior_median_ratio,is_10x_prior_median,has_prior_normal_transaction,outside_trans_hours_80,is_online,risk_time_22_04,interact_repeat_category,merchant_change_count,rolling_sum_amt_1h,is_high_amt
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.970000,0,0,0.000000,1,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,30,NaN,NaN,0,0,0,1,1,0.000000,0,4.970000,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.230000,0,0,0.000000,1,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,40,NaN,NaN,0,0,0,0,1,0.000000,0,107.230000,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.110000,0,0,0.000000,1,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,56,NaN,NaN,0,0,0,0,1,0.000000,0,220.110000,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.000000,0,0,0.000000,1,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,51,NaN,NaN,0,0,0,0,1,0.000000,0,45.000000,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.960000,0,0,0.000000,1,1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,32,NaN,NaN,0,0,0,0,1,0.000000,0,41.960000,0



타깃 분포:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Fraud Rate (%): 0.5788651743883394


In [8]:
# ==========================================
# STEP 6. Feature Set 1~5 정의
# ==========================================

set1 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min"
]


set2 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "merchant_change_count"
]


set3 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "high_speed"
]


set4 = [
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "category",
    "amt",
    "trans_hour",
    "age"
]


set5 = [
    "category",
    "amt",
    "is_online",
    "recent_24h_high_amt_count",
    "category_recent_fraud_rate",
    "speed_2",
    "customer_mean_amt",
    "customer_std_amt",
    "amt_ratio_to_mean",
    "amt_zscore_card",
    "customer_transaction_count",
    "trans_hour",
    "age",
    "rolling_sum_amt_1h",
    "prior_normal_median_amt",
    "amt_to_prior_median_ratio",
    "risk_time_22_04",
    "interact_repeat_category",
    "has_prior_normal_transaction"
]


feature_sets = {
    "Set 1": set1,
    "Set 2": set2,
    "Set 3": set3,
    "Set 4": set4,
    "Set 5": set5
}


for name, features in feature_sets.items():
    print(
        f"{name}: {len(features)}개 변수"
    )

Set 1: 10개 변수
Set 2: 11개 변수
Set 3: 11개 변수
Set 4: 6개 변수
Set 5: 19개 변수


In [9]:
# ==========================================
# STEP 7. 변수 존재 여부 확인
# ==========================================

print("===== 변수 존재 여부 확인 =====")

for set_name, features in feature_sets.items():

    missing = [
        feature
        for feature in features
        if feature not in df.columns
    ]

    if len(missing) == 0:

        print(
            f"✅ {set_name}: "
            f"모든 변수 존재 ({len(features)}개)"
        )

    else:

        print(
            f"❌ {set_name}: "
            f"누락 변수 = {missing}"
        )

===== 변수 존재 여부 확인 =====
✅ Set 1: 모든 변수 존재 (10개)
✅ Set 2: 모든 변수 존재 (11개)
✅ Set 3: 모든 변수 존재 (11개)
✅ Set 4: 모든 변수 존재 (6개)
✅ Set 5: 모든 변수 존재 (19개)


In [10]:
# ==========================================
# STEP 8. 결측치 확인
# ==========================================

all_model_features = sorted(
    set(
        feature
        for features in feature_sets.values()
        for feature in features
    )
)

missing_summary = (
    df[all_model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("===== 결측치가 있는 변수 =====")

display(
    missing_summary[
        missing_summary > 0
    ]
)

===== 결측치가 있는 변수 =====


,0
amt_to_prior_median_ratio,1649
prior_normal_median_amt,1649


In [11]:
# ==========================================
# STEP 9. 전처리 + Logistic Regression
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


CATEGORICAL_COLUMNS = [
    "category"
]


def make_logistic_pipeline(
    features,
    C=1.0,
    penalty="l2",
    solver="lbfgs"
):

    categorical_features = [
        col
        for col in features
        if col in CATEGORICAL_COLUMNS
    ]

    numeric_features = [
        col
        for col in features
        if col not in CATEGORICAL_COLUMNS
    ]


    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])


    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ])


    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ])


    model = LogisticRegression(
        C=C,
        penalty=penalty,
        solver=solver,
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )


    pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ])

    return pipeline

In [12]:
# ==========================================
# STEP 10. 평가 함수
# ==========================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

THRESHOLD = 0.9


def evaluate_binary_model(
    y_true,
    prob,
    threshold=0.9
):

    pred = (
        prob >= threshold
    ).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1]
    ).ravel()


    return {

        "PR-AUC":
            average_precision_score(
                y_true,
                prob
            ),

        "ROC-AUC":
            roc_auc_score(
                y_true,
                prob
            ),

        "Accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "Precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "FP": fp,
        "FN": fn,
        "TP": tp,
        "TN": tn
    }

In [13]:
# ==========================================
# STEP 11. 시간순 80:20 성능 비교
# ==========================================

results_80 = []

# 시간순 80:20 분할
split_idx_80 = int(len(df) * 0.8)

train_df_80 = df.iloc[:split_idx_80].copy()
val_df_20 = df.iloc[split_idx_80:].copy()

print("===== 시간순 80:20 분할 =====")
print("Train:", train_df_80.shape)
print("Validation:", val_df_20.shape)

print("\nTrain 기간:")
print(
    train_df_80["trans_date_trans_time"].min(),
    "~",
    train_df_80["trans_date_trans_time"].max()
)

print("\nValidation 기간:")
print(
    val_df_20["trans_date_trans_time"].min(),
    "~",
    val_df_20["trans_date_trans_time"].max()
)

print("\nTrain Fraud Rate:")
print(
    train_df_80["is_fraud"].mean() * 100,
    "%"
)

print("\nValidation Fraud Rate:")
print(
    val_df_20["is_fraud"].mean() * 100,
    "%"
)


# ==========================================
# Feature Set 1~5 학습
# ==========================================

for set_name, features in feature_sets.items():

    print("\n" + "=" * 70)
    print(f"80:20 | {set_name}")
    print("=" * 70)

    X_train = train_df_80[features]
    y_train = train_df_80["is_fraud"]

    X_val = val_df_20[features]
    y_val = val_df_20["is_fraud"]

    model = make_logistic_pipeline(
        features=features,
        C=1.0,
        penalty="l2",
        solver="lbfgs"
    )

    model.fit(
        X_train,
        y_train
    )

    val_prob = model.predict_proba(
        X_val
    )[:, 1]

    metrics = evaluate_binary_model(
        y_true=y_val,
        prob=val_prob,
        threshold=THRESHOLD
    )

    results_80.append({
        "Feature Set": set_name,
        "N Features": len(features),
        "Threshold": THRESHOLD,
        **metrics
    })

    print(f"PR-AUC    : {metrics['PR-AUC']:.6f}")
    print(f"ROC-AUC   : {metrics['ROC-AUC']:.6f}")
    print(f"Precision : {metrics['Precision']:.6f}")
    print(f"Recall    : {metrics['Recall']:.6f}")
    print(f"F1        : {metrics['F1']:.6f}")
    print(
        f"TP={metrics['TP']:,} | "
        f"FP={metrics['FP']:,} | "
        f"FN={metrics['FN']:,} | "
        f"TN={metrics['TN']:,}"
    )

===== 시간순 80:20 분할 =====
Train: (1037340, 31)
Validation: (259335, 31)

Train 기간:
2019-01-01 00:00:18 ~ 2020-03-06 07:15:17

Validation 기간:
2020-03-06 07:16:43 ~ 2020-06-21 12:13:37

Train Fraud Rate:
0.5753176393467908 %

Validation Fraud Rate:
0.5930553145545336 %

80:20 | Set 1
PR-AUC    : 0.447580
ROC-AUC   : 0.971735
Precision : 0.359063
Recall    : 0.747074
F1        : 0.485015
TP=1,149 | FP=2,051 | FN=389 | TN=255,746

80:20 | Set 2
PR-AUC    : 0.451172
ROC-AUC   : 0.971848
Precision : 0.363295
Recall    : 0.754226
F1        : 0.490383
TP=1,160 | FP=2,033 | FN=378 | TN=255,764

80:20 | Set 3
PR-AUC    : 0.447362
ROC-AUC   : 0.971674
Precision : 0.358367
Recall    : 0.747724
F1        : 0.484517
TP=1,150 | FP=2,059 | FN=388 | TN=255,738

80:20 | Set 4
PR-AUC    : 0.417983
ROC-AUC   : 0.970308
Precision : 0.320330
Recall    : 0.731469
F1        : 0.445545
TP=1,125 | FP=2,387 | FN=413 | TN=255,410

80:20 | Set 5
PR-AUC    : 0.545416
ROC-AUC   : 0.985219
Precision : 0.509146
Recall 

In [14]:
# ==========================================
# STEP 12. 80:20 성능 순위
# ==========================================

results_80_df = pd.DataFrame(
    results_80
)

results_80_df = (
    results_80_df
    .sort_values(
        by=[
            "PR-AUC",
            "F1",
            "Recall"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

results_80_df.insert(
    0,
    "Rank",
    range(1, len(results_80_df) + 1)
)

display(
    results_80_df[
        [
            "Rank",
            "Feature Set",
            "N Features",
            "PR-AUC",
            "Precision",
            "Recall",
            "F1",
            "FP",
            "FN"
        ]
    ]
)

,Rank,Feature Set,N Features,PR-AUC,Precision,Recall,F1,FP,FN
0,1,Set 5,19,0.545416,0.509146,0.760078,0.609807,1127,369
1,2,Set 2,11,0.451172,0.363295,0.754226,0.490383,2033,378
2,3,Set 1,10,0.447580,0.359063,0.747074,0.485015,2051,389
3,4,Set 3,11,0.447362,0.358367,0.747724,0.484517,2059,388
4,5,Set 4,6,0.417983,0.320330,0.731469,0.445545,2387,413


In [15]:
# ==========================================
# STEP 13. 3-Fold 날짜 정의
# ==========================================

folds = {

    "Fold 1": {
        "train_start": "2019-01-01",
        "train_end":   "2019-09-30 23:59:59",
        "val_start":   "2019-10-01",
        "val_end":     "2019-12-31 23:59:59"
    },

    "Fold 2": {
        "train_start": "2019-01-01",
        "train_end":   "2019-12-31 23:59:59",
        "val_start":   "2020-01-01",
        "val_end":     "2020-03-31 23:59:59"
    },

    "Fold 3": {
        "train_start": "2019-01-01",
        "train_end":   "2020-03-31 23:59:59",
        "val_start":   "2020-04-01",
        "val_end":     df["trans_date_trans_time"].max()
    }
}


def get_fold_data(
    data,
    period
):

    train_mask = (
        (
            data["trans_date_trans_time"]
            >= pd.Timestamp(
                period["train_start"]
            )
        )
        &
        (
            data["trans_date_trans_time"]
            <= pd.Timestamp(
                period["train_end"]
            )
        )
    )

    val_mask = (
        (
            data["trans_date_trans_time"]
            >= pd.Timestamp(
                period["val_start"]
            )
        )
        &
        (
            data["trans_date_trans_time"]
            <= pd.Timestamp(
                period["val_end"]
            )
        )
    )

    fold_train = data.loc[
        train_mask
    ].copy()

    fold_val = data.loc[
        val_mask
    ].copy()

    return fold_train, fold_val

In [16]:
fold_check = []

for fold_name, period in folds.items():

    fold_train, fold_val = get_fold_data(
        df,
        period
    )

    fold_check.append({

        "Fold": fold_name,

        "Train Transactions":
            len(fold_train),

        "Train Fraud":
            int(
                fold_train["is_fraud"].sum()
            ),

        "Validation Transactions":
            len(fold_val),

        "Validation Fraud":
            int(
                fold_val["is_fraud"].sum()
            ),

        "Validation Fraud Rate (%)":
            (
                fold_val["is_fraud"].mean()
                * 100
            )
    })


fold_check_df = pd.DataFrame(
    fold_check
)

display(
    fold_check_df
)

,Fold,Train Transactions,Train Fraud,Validation Transactions,Validation Fraud,Validation Fraud Rate (%)
0,Fold 1,644611,3786,280239,1434,0.511706
1,Fold 2,924850,5220,172843,1123,0.649723
2,Fold 3,1097693,6343,198982,1163,0.584475


In [17]:
# ==========================================
# STEP 14. 3-Fold 시간순 교차검증
# ==========================================

cv_results = []


for set_name, features in feature_sets.items():

    print("\n" + "=" * 80)
    print(set_name)
    print("=" * 80)

    for fold_name, period in folds.items():

        fold_train, fold_val = get_fold_data(
            df,
            period
        )

        X_train = fold_train[
            features
        ]

        y_train = fold_train[
            "is_fraud"
        ]

        X_val = fold_val[
            features
        ]

        y_val = fold_val[
            "is_fraud"
        ]

        model = make_logistic_pipeline(
            features=features,
            C=1.0,
            penalty="l2",
            solver="lbfgs"
        )

        model.fit(
            X_train,
            y_train
        )

        val_prob = model.predict_proba(
            X_val
        )[:, 1]

        metrics = evaluate_binary_model(
            y_true=y_val,
            prob=val_prob,
            threshold=THRESHOLD
        )

        cv_results.append({

            "Feature Set":
                set_name,

            "Fold":
                fold_name,

            "Threshold":
                THRESHOLD,

            **metrics
        })

        print(
            f"{fold_name} | "
            f"PR-AUC={metrics['PR-AUC']:.6f} | "
            f"Precision={metrics['Precision']:.6f} | "
            f"Recall={metrics['Recall']:.6f} | "
            f"F1={metrics['F1']:.6f}"
        )


Set 1
Fold 1 | PR-AUC=0.413515 | Precision=0.253690 | Recall=0.767085 | F1=0.381282
Fold 2 | PR-AUC=0.488513 | Precision=0.419435 | Recall=0.753339 | F1=0.538854
Fold 3 | PR-AUC=0.436070 | Precision=0.345433 | Recall=0.744626 | F1=0.471935

Set 2
Fold 1 | PR-AUC=0.414701 | Precision=0.256422 | Recall=0.772664 | F1=0.385056
Fold 2 | PR-AUC=0.489921 | Precision=0.423500 | Recall=0.754230 | F1=0.542427
Fold 3 | PR-AUC=0.438464 | Precision=0.349581 | Recall=0.752365 | F1=0.477360

Set 3
Fold 1 | PR-AUC=0.412318 | Precision=0.253101 | Recall=0.768480 | F1=0.380788
Fold 2 | PR-AUC=0.487979 | Precision=0.420686 | Recall=0.753339 | F1=0.539885
Fold 3 | PR-AUC=0.436036 | Precision=0.345404 | Recall=0.746346 | F1=0.472252

Set 4
Fold 1 | PR-AUC=0.382927 | Precision=0.220259 | Recall=0.747559 | F1=0.340263
Fold 2 | PR-AUC=0.460078 | Precision=0.383871 | Recall=0.741763 | F1=0.505922
Fold 3 | PR-AUC=0.408456 | Precision=0.309793 | Recall=0.731728 | F1=0.435294

Set 5
Fold 1 | PR-AUC=0.513712 | Pr

In [18]:
cv_results_df = pd.DataFrame(
    cv_results
)

display(
    cv_results_df
)

,Feature Set,Fold,Threshold,PR-AUC,ROC-AUC,Accuracy,Precision,Recall,F1,FP,FN,TP,TN
0,Set 1,Fold 1,0.900000,0.413515,0.969926,0.987261,0.253690,0.767085,0.381282,3236,334,1100,275569
1,Set 1,Fold 2,0.900000,0.488513,0.975430,0.991622,0.419435,0.753339,0.538854,1171,277,846,170549
2,Set 1,Fold 3,0.900000,0.436070,0.971420,0.990260,0.345433,0.744626,0.471935,1641,297,866,196178
3,Set 2,Fold 1,0.900000,0.414701,0.970238,0.987371,0.256422,0.772664,0.385056,3213,326,1108,275592
4,Set 2,Fold 2,0.900000,0.489921,0.975698,0.991732,0.423500,0.754230,0.542427,1153,276,847,170567
5,Set 2,Fold 3,0.900000,0.438464,0.971672,0.990371,0.349581,0.752365,0.477360,1628,288,875,196191
6,Set 3,Fold 1,0.900000,0.412318,0.969870,0.987211,0.253101,0.768480,0.380788,3252,332,1102,275553
7,Set 3,Fold 2,0.900000,0.487979,0.975386,0.991657,0.420686,0.753339,0.539885,1165,277,846,170555
8,Set 3,Fold 3,0.900000,0.436036,0.971441,0.990250,0.345404,0.746346,0.472252,1645,295,868,196174
9,Set 4,Fold 1,0.900000,0.382927,0.968102,0.985166,0.220259,0.747559,0.340263,3795,362,1072,275010


In [19]:
# ==========================================
# STEP 15. 3-Fold 평균 / 표준편차
# ==========================================

cv_summary_df = (
    cv_results_df
    .groupby(
        "Feature Set"
    )
    .agg(

        Mean_PR_AUC=(
            "PR-AUC",
            "mean"
        ),

        Std_PR_AUC=(
            "PR-AUC",
            "std"
        ),

        Mean_ROC_AUC=(
            "ROC-AUC",
            "mean"
        ),

        Mean_Precision=(
            "Precision",
            "mean"
        ),

        Mean_Recall=(
            "Recall",
            "mean"
        ),

        Mean_F1=(
            "F1",
            "mean"
        ),

        Total_FP=(
            "FP",
            "sum"
        ),

        Total_FN=(
            "FN",
            "sum"
        ),

        Total_TP=(
            "TP",
            "sum"
        ),

        Total_TN=(
            "TN",
            "sum"
        )
    )
    .reset_index()
)


# ==========================================
# 순위 기준
# 1. Mean PR-AUC 높은 순
# 2. Std PR-AUC 낮은 순
# 3. Mean F1 높은 순
# ==========================================

cv_summary_df = (
    cv_summary_df
    .sort_values(
        by=[
            "Mean_PR_AUC",
            "Std_PR_AUC",
            "Mean_F1"
        ],
        ascending=[
            False,
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


cv_summary_df.insert(
    0,
    "Rank",
    range(
        1,
        len(cv_summary_df) + 1
    )
)


display(
    cv_summary_df[
        [
            "Rank",
            "Feature Set",
            "Mean_PR_AUC",
            "Std_PR_AUC",
            "Mean_Precision",
            "Mean_Recall",
            "Mean_F1",
            "Total_FP",
            "Total_FN"
        ]
    ]
)

,Rank,Feature Set,Mean_PR_AUC,Std_PR_AUC,Mean_Precision,Mean_Recall,Mean_F1,Total_FP,Total_FN
0,1,Set 5,0.540066,0.030566,0.491539,0.761604,0.595271,3091,884
1,2,Set 2,0.447695,0.038450,0.343168,0.759753,0.468281,5994,890
2,3,Set 1,0.446033,0.038479,0.339519,0.755017,0.464024,6048,908
3,4,Set 3,0.445444,0.038698,0.339730,0.756055,0.464308,6062,904
4,5,Set 4,0.417154,0.039304,0.304641,0.740350,0.427160,7028,964


In [20]:
# ==========================================
# STEP 16-1. 최적 Feature Set 선택
# ==========================================

best_set_name = (
    cv_summary_df
    .iloc[0][
        "Feature Set"
    ]
)

best_features = (
    feature_sets[
        best_set_name
    ]
)

print(
    "최적 Feature Set:",
    best_set_name
)

print(
    "변수 개수:",
    len(best_features)
)

print(
    "변수:",
    best_features
)

최적 Feature Set: Set 5
변수 개수: 19
변수: ['category', 'amt', 'is_online', 'recent_24h_high_amt_count', 'category_recent_fraud_rate', 'speed_2', 'customer_mean_amt', 'customer_std_amt', 'amt_ratio_to_mean', 'amt_zscore_card', 'customer_transaction_count', 'trans_hour', 'age', 'rolling_sum_amt_1h', 'prior_normal_median_amt', 'amt_to_prior_median_ratio', 'risk_time_22_04', 'interact_repeat_category', 'has_prior_normal_transaction']


In [21]:
# ==========================================
# STEP 16-2. Fold 3 Train으로 최종 계수 확인
# ==========================================

fold3_train, fold3_val = get_fold_data(
    df,
    folds["Fold 3"]
)

X_train_best = fold3_train[
    best_features
]

y_train_best = fold3_train[
    "is_fraud"
]


best_model = make_logistic_pipeline(
    features=best_features,
    C=1.0,
    penalty="l2",
    solver="lbfgs"
)


best_model.fit(
    X_train_best,
    y_train_best
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['amt', 'is_online',
                                                   'recent_24h_high_amt_count',
                                                   'category_recent_fraud_rate',
                                                   'speed_2',
                                                   'customer_mean_amt',
                                                   'customer_std_amt',
                                                   'amt_ratio_to_mean',
                                                   'amt_zscore_card',
                                                   'customer_transaction_cou...
                                                   'prior_normal_median_amt',
                                                   'amt_to_prior_median_ratio',
                                                   'risk_time_22_04',
                                                   'interact_repeat_category',
                                                   'has_prior_normal_transaction']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['category'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42))])

In [22]:
# ==========================================
# STEP 16-3. 전처리 후 변수명
# ==========================================

feature_names = (
    best_model
    .named_steps[
        "preprocessor"
    ]
    .get_feature_names_out()
)


coefficients = (
    best_model
    .named_steps[
        "model"
    ]
    .coef_[0]
)

In [23]:
coef_df = pd.DataFrame({

    "Feature":
        feature_names,

    "Coefficient":
        coefficients
})


# Odds Ratio
coef_df[
    "Odds Ratio"
] = np.exp(
    coef_df[
        "Coefficient"
    ]
)


# 계수 절댓값
coef_df[
    "Abs Coefficient"
] = (
    coef_df[
        "Coefficient"
    ]
    .abs()
)


# 영향력이 큰 순서
coef_df = (
    coef_df
    .sort_values(
        "Abs Coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    coef_df.head(30)
)

,Feature,Coefficient,Odds Ratio,Abs Coefficient
0,cat__category_gas_transport,2.879288,17.801592,2.879288
1,cat__category_grocery_net,2.313841,10.113192,2.313841
2,cat__category_shopping_net,-2.061457,0.127268,2.061457
3,cat__category_grocery_pos,2.016020,7.508384,2.016020
4,cat__category_misc_pos,1.502445,4.492659,1.502445
5,cat__category_shopping_pos,-1.404690,0.245443,1.404690
6,cat__category_personal_care,1.361199,3.900869,1.361199
7,cat__category_kids_pets,1.279083,3.593344,1.279083
8,cat__category_food_dining,1.066905,2.906371,1.066905
9,num__risk_time_22_04,1.043998,2.840551,1.043998


In [24]:
print(
    "===== Fraud 방향 TOP 15 ====="
)

fraud_direction = (
    coef_df[
        coef_df[
            "Coefficient"
        ] > 0
    ]
    .sort_values(
        "Coefficient",
        ascending=False
    )
    .head(15)
)

display(
    fraud_direction
)

===== Fraud 방향 TOP 15 =====


,Feature,Coefficient,Odds Ratio,Abs Coefficient
0,cat__category_gas_transport,2.879288,17.801592,2.879288
1,cat__category_grocery_net,2.313841,10.113192,2.313841
3,cat__category_grocery_pos,2.016020,7.508384,2.016020
4,cat__category_misc_pos,1.502445,4.492659,1.502445
6,cat__category_personal_care,1.361199,3.900869,1.361199
7,cat__category_kids_pets,1.279083,3.593344,1.279083
8,cat__category_food_dining,1.066905,2.906371,1.066905
9,num__risk_time_22_04,1.043998,2.840551,1.043998
10,num__amt,1.009524,2.744295,1.009524
12,cat__category_health_fitness,0.870294,2.387612,0.870294


In [25]:
print(
    "===== Normal 방향 TOP 15 ====="
)

normal_direction = (
    coef_df[
        coef_df[
            "Coefficient"
        ] < 0
    ]
    .sort_values(
        "Coefficient",
        ascending=True
    )
    .head(15)
)

display(
    normal_direction
)

===== Normal 방향 TOP 15 =====


,Feature,Coefficient,Odds Ratio,Abs Coefficient
2,cat__category_shopping_net,-2.061457,0.127268,2.061457
5,cat__category_shopping_pos,-1.404690,0.245443,1.404690
11,cat__category_misc_net,-0.874473,0.417082,0.874473
16,num__customer_transaction_count,-0.404195,0.667514,0.404195
20,num__customer_std_amt,-0.218821,0.803465,0.218821
22,num__interact_repeat_category,-0.164933,0.847951,0.164933
24,num__has_prior_normal_transaction,-0.132388,0.876001,0.132388
26,num__prior_normal_median_amt,-0.092578,0.911578,0.092578
